# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Miriam Wepiya Gale
**Student ID:** 46582028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [2]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

#  --- Local (with a .env file) ---
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

#  --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [3]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
   response = client.chat.completions.create(
       model=MODEL,
       messages=[
          {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
      ],
       temperature=temperature,
        max_tokens=max_tokens,
    )
   return response.choices[0].message.content , response.usage
#
# TODO: Call it once with a simple question and print the answer.
question , usage = ask_llm("How does the internet work?")

print(question)
# TODO: Print response.usage as well — how many tokens did your call consume?
print(usage)

The internet is a complex system, but I'll try to break it down in a simple way.

**The Basics**

The internet is a global network of interconnected computers and servers that communicate with each other using standardized protocols. It's like a huge, virtual web of information that allows devices to share and access data.

**Key Components**

1. **Internet Service Providers (ISPs)**: These are companies that provide access to the internet. They connect your device to the internet using physical infrastructure like cables, fiber optics, or wireless networks.
2. **Internet Protocol (IP) Addresses**: Each device on the internet has a unique IP address, which is like a street address for your device. This allows devices to find and communicate with each other.
3. **Packet Switching**: When you send data over the internet, it's broken into small packets of data. Each packet is given a header with the sender's and receiver's IP addresses, and then it's transmitted independently. The packets

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** 
##
* 1
The system role  gives the LLM instructions about how it should behave or what role it should take.

Example: "You are a helpful assistant. Explain the concepts in simple language"

The user role contains the actual request or question that we want the LLM to answer.

Example: "Explain how the internet works?"

* 2

A token is a small piece of text that  LLM processes. API providers bill per token because longer prompts and responses require more computation, so the token usage better reflects the amount of work the model performs.

### Part 1.2 — Temperature: the randomness dial

In [4]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.

question_2 = "Suggest a name for a savings product for market traders in Accra."

answers_0 = []

for i in range(5):
  answer, usage = ask_llm(question_2, temperature=0.0)
  answers_0.append(answer)
print(len(answers_0))


answers_1_2 = []
for i in range(5):
  answer,usage = ask_llm(question_2, temperature=1.2)
  answers_1_2.append(answer)
print(len(answers_1_2))


print("Temperature 0.0")
for answer in answers_0:
  print(answer)

print("Temperature 1.2")
for answer in answers_1_2:
  print(answer)
  
      


#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

5
5
Temperature 0.0
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", which could appeal to market traders in Accra.
4. **Market Mobi**: This name incorporates "mobi", short for mobile, to suggest a convenient and accessible savings product.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "save" or "keep", making this name straightforward and easy to understand.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Adanfo Account**: "Adanfo" is a Ghanaian word for "friends" or "partners", suggesting a savings product that supports and partners with ma

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** 

* 1

At both temperatures, the answers were similar, but the key difference was how the answers were being framed and presented. The temperature of 1.2 produced more variation in the wording of the answers while the temperature of 0.0 was more consistent.

For a loan decision support system, a lower temperature such as 0.0 would be more appropriate because the system would give more consistent responses rather than highly varied outputs.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [5]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [6]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

SUMMARY_PROMPT_V1 = "Summarise this: "

letter_text_002 = LETTERS["L002"]
result_002 = ask_llm(SUMMARY_PROMPT_V1 + letter_text_002)
print(result_002)

letter_text_006 = LETTERS["L006"]
result_006 = ask_llm(SUMMARY_PROMPT_V1 + letter_text_006)
print(result_006)

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)

SUMMARY_PROMPT_V2_SYSTEM = "You are an assistant to a microfinance loan officer. Summarize loan applications factually and neutrally. Do not invent or assume any details that are not stated in the application. Keep the summary to 3–4 sentences."

#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"

user_prompt_002 = f"Summarize this loan application:\n\n{letter_text_002}"

user_prompt_006 = f"Summarize this loan application:\n\n{letter_text_006}"

#   Run V2 on the same two letters at temperature=0.
result_prompt_002 = ask_llm(user_prompt_002, SUMMARY_PROMPT_V2_SYSTEM, temperature=0.0)

result_prompt_006 = ask_llm(user_prompt_006, SUMMARY_PROMPT_V2_SYSTEM, temperature=0.0)


# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("V1 - L002")
print(result_prompt_002)

print("V2 - L006")
print(result_prompt_006)



("Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's struggling due to slow business, but expects it to improve after the festive season. He has no collateral to offer, but promises to repay the loan as soon as possible.", CompletionUsage(completion_tokens=71, prompt_tokens=133, total_tokens=204, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041803799, prompt_time=0.015765234, completion_time=0.251075025, total_time=0.266840259))
('Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and promises to repay the loan in one year when his businesses are successful.', CompletionUsage(completion_tokens=73, prompt_tokens=135, total_tokens=208, completion_tokens_details=None, prompt_tokens_details=None

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** 

* 1

V1 did not give the model clear instruction about its role, tone, constraints or length of the summary. For example, in L002, V1 described Kwame as "seeking urgent assistance with the loan", which is more interpretive than simply presenting the application facts. V2 was more controlled and neutral, summarising the application in concise sentences.
For L006, V1 included "offering his trustworthiness as assurance", while V2 stated more neutrally that he "asserts that he is trustworthy." 
V2 also followed the requested 3-4 sentence structure.


* 2

“No invented details” is essential because the loan decision-support system must be grounded in the applicant's actual information. If the LLM invents or assumes information, it could create a false impression of the applicant and potentially lead to an unfair loan decision. This failure mode is called an LLM hallucination.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [7]:
import json
import pandas as pd
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON 
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)

EXTRACT_PROMPT = """
Extract the required information from the loan application.
Return ONLY a JSON object with EXACTLY these keys:
applicant_name (string),
amount_ghs (number),
purpose (string),
monthly_profit_ghs (number or null),
has_collateral_or_guarantor (boolean),
repayment_months (number or null).

If a field is not stated in the letter, use null. Do not guess.

Example letter:
My name is Ama Mensah. I need GHS 10,000 to buy a freezer
for my beverage business. I make GHS 800 profit each month.
My brother will guarantee the loan, and I will repay it over 12 months.

Example JSON:
{
  "applicant_name": "Ama Mensah",
  "amount_ghs": 10000,
  "purpose": "buy a freezer for my beverage business",
  "monthly_profit_ghs": 800,
  "has_collateral_or_guarantor": true,
  "repayment_months": 12
}

"""
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,

  # TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON 
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)

#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
    prompt = f"{EXTRACT_PROMPT}\n\nLoan application:\n{letter_text}"
    response, usage = ask_llm(prompt, temperature=0.0)
    response = response.replace("```json", "").replace("```", "").strip()

    try:
       data = json.loads(response)
       return data
    except json.JSONDecodeError:
        print("Warning: Could not parse the LLM response as JSON")
        return None



# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

results = []
for letter_id, letter_text in LETTERS.items():
    result = extract_fields(letter_text)
    results.append(result)

print(len(results))

df = pd.DataFrame(results)
display(df)

print(df.columns.tolist())

6


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,Yaw Owusu,12000,for my poultry farm at Nsawam for feed and 500...,1500.0,True,18.0
4,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


['applicant_name', 'amount_ghs', 'purpose', 'monthly_profit_ghs', 'has_collateral_or_guarantor', 'repayment_months']


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** 

* 1

The example should not come from the six letters because the six letters are the data we are actually testing the extraction system on. If one of them were used as the example, the model could simply follow or memorize that example instead of demonstrating that the prompt works correctly on new, unseen applications. Using a separate example tests whether the extraction instructions generalize to other loan applications.

* 2

The instruction is important because some information is missing from the loan letters. Without the instruction, an LLM may try to fill in missing information based on assumptions rather than the actual application. For example, Kwame's application did not state his monthly profit or a specific repayment period, and our extraction correctly returned None for both fields. This prevents the system from creating false information that could affect a loan decision being made.

* 3

Temperature 0 is appropriate for extraction because we want the model to produce consistent and predictable results from the same information. For example, the same loan application should consistently produce the same applicant name, loan amount, and other fields. For creative tasks, however, variation can be useful, so a higher temperature can produce more diverse ideas and wording.


### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [12]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

def BRIEF_PROMPT(letter_text, result ):
  prompt =  f"""
Review this loan application and the extracted information.

Loan application:
{letter_text}

Extracted information:
{result}

Strengths: 

- Identify and list strengths that are directly supported by the loan application.
- Do not invent or assume any information.

Risks / red flags:

- Idntify risks or red flags that are being supported by the loan application.

- Do not invent or assume any information.

Missing information:
- Identify and list important information that is not being provided in the application and that the loan officer should request them.

- Do not invent or assume any information.

Suggested next step:

- Recommend an appropriate next step based on the information available.

- Examples include "invite for interview", "request documents", or "flag for senior review".

- Do NOT recommend approving or rejecting the loan.

Final decision: 

- The final loan decision must be made by a licensed human loan officer.

- Do not approve or reject the application.
"""
  return prompt

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

briefs = {}
for letter_id, letter_text in LETTERS.items():
  result = results[list(LETTERS.keys()).index(letter_id)]
  prompt = BRIEF_PROMPT(letter_text, result)
  brief, usage= ask_llm(prompt, temperature=0.0)
  briefs[letter_id] = brief

print("====== L001 =====")
print(briefs["L001"])

print("\n====== L002 =====")
print(briefs["L002"])

print("\n====== L006 =====")
print(briefs["L006"])


====== L001 =====
**Review of Loan Application**

**Extracted Information:**
The extracted information is accurate and includes the following key points:
- Applicant name: Akosua Mensah
- Loan amount: GHS 8,000
- Purpose: Buy a deep freezer and expand into frozen foods
- Monthly profit: GHS 900
- Has collateral or guarantor: True
- Repayment months: 20

**Strengths:**
1. The applicant has a stable business, having sold provisions at Makola Market for 12 years.
2. The applicant has a regular monthly profit of GHS 900.
3. The applicant has saved GHS 2,500 with the susu scheme over two years without missing a contribution, demonstrating financial discipline.
4. The applicant has a guarantor, her sister, who is a teacher, providing an added layer of security.

**Risks / Red Flags:**
1. The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit of GHS 900, which might pose a repayment risk.
2. The repayment plan of GHS 450 over 20 months may be challenging if the

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** 

* 1

Yes. For L003, the system should identify strengths such as the applicant's registered business, existing apprentices, monthly profit of GHS 2,800, fixed deposit of GHS 5,000 that can be pledged, and a clear repayment plan of GHS 1,100 for 15 months. These are strong indicators because they are directly supported by the application. For L006, the important red flags are that Kofi has not started any of the three proposed businesses, has no previous business experience or collateral, and plans to repay the GHS 50,000 loan when the businesses become successful. Therefore, the system should highlight the lack of established business activity and security as risks.

* 2

We forbid the model from saying "approve" or "reject" because the system is intended to support, not replace, the human loan officer. Practically, a human officer needs to verify documents and consider information that the model may not have access to. Ethically, allowing an AI to make the final decision could lead to unfair or incorrect lending decisions and makes it harder to hold a human decision-maker accountable.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 

abbe7ca

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [9]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

### Part 4.2 — Reliability: is the system consistent?

In [10]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [11]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.